# Ghost Envelope

Notebook-native smoke reproduction of the ghost envelope visualization. The code below runs the per-architecture snapshot logic directly and then renders the envelope plot.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys


def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "ghosts").exists() and (
            (base / "tutorials").exists() or (base / "experiments").exists()
        ):
            return base

    if in_colab():
        repo = Path('/content/ghosts-of-softmax')
        if not repo.exists():
            subprocess.run(
                [
                    'git', 'clone', '--depth', '1',
                    'https://github.com/piyush314/ghosts-of-softmax.git',
                    str(repo),
                ],
                check=True,
            )
        return repo

    raise RuntimeError(
        'Run this notebook from inside the ghosts-of-softmax repository, '
        'or open it in Google Colab so the setup cell can clone the repo automatically.'
    )


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_module(name, relative_path):
    path = REPO / relative_path
    module_dir = str(path.parent)
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


OUTPUT_ROOT = Path('/tmp/ghosts-of-softmax-notebooks')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO}")
print(f"Notebook outputs: {OUTPUT_ROOT}")


In [ ]:
from IPython.display import Image, display
import numpy as np
import torch

ghost = load_module('fig_ghostenv_run', 'experiments/ghostenvelope/run.py')

X, y = ghost.loadDigits()
Xtr, _, ytr, _ = ghost.stratSplit(X, y)
Xtr = torch.tensor(Xtr, dtype=torch.float32)
ytr = torch.tensor(ytr, dtype=torch.long)

rng = np.random.default_rng(42)
idx = rng.choice(len(Xtr), size=min(256, len(Xtr)), replace=False)
Xeval, yeval = Xtr[idx], ytr[idx]

arch_names = ['Linear', 'MLP']
seeds = [0]
data = {}
for arch_name in arch_names:
    cls, lr, steps = ghost.ARCHS[arch_name]
    data[arch_name] = {}
    for seed in seeds:
        data[arch_name][seed] = ghost.runArch(
            arch_name,
            cls,
            lr,
            20,
            Xtr,
            ytr,
            Xeval,
            yeval,
            seed,
        )

payload = {
    'config': {'seeds': seeds, 'smoke': True},
    'data': data,
    'archs': arch_names,
}
out_dir = OUTPUT_ROOT / 'ghostenvelope'
out_dir.mkdir(parents=True, exist_ok=True)
out_png = out_dir / 'ghost-envelope.png'
out_pdf = out_dir / 'ghost-envelope.pdf'

ghost.make_plot(payload, out_png, out_pdf)
display(Image(filename=str(out_png)))

ghost.build_summary(
    payload['data'],
    payload['archs'],
    payload['config'],
    out_dir / 'ghostenvelope.pt',
    out_png,
    out_pdf,
    out_dir / 'summary.json',
)
